# Hey Igor microWakeWord Training

Clean Colab notebook for training and exporting an ESPHome-compatible `micro_wake_word` model.

Notes:
- Background audio prep avoids `datasets.Audio()` decoding to stay compatible with current Colab runtimes.
- Positive examples are trained through `type: "clips"`.
- Built-in validation/export in this `micro-wake-word` version is not reliable with `clips`-based datasets, so training and export are split.
- For a real training run, use a Colab GPU runtime.


In [1]:
# 1. Test one generated pronunciation

import os
import sys
import importlib.util
from functools import wraps
from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !cd piper-sample-generator && git checkout 213d4d5

model_path = "piper-sample-generator/models/en_US-libritts_r-medium.pt"
if not os.path.exists(model_path):
    !wget -O {model_path} "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"

!pip install -q piper-tts piper-phonemize-cross webrtcvad

if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

import torch

_original_torch_load = torch.load

@wraps(_original_torch_load)
def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = _torch_load_compat

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

from generate_samples import generate_samples

target_word = "hey___ee__gore"  # @param {type:"string"}

def text_to_speech(text):
    generate_samples(
        text=text,
        max_samples=1,
        length_scales=[1.1],
        noise_scales=[0.7],
        noise_scale_ws=[0.7],
        output_dir="./",
        batch_size=1,
        auto_reduce_batch_size=True,
        file_names=["test_generation.wav"],
    )

text_to_speech(target_word)
Audio("test_generation.wav", autoplay=True)


Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 161 (delta 64), reused 62 (delta 50), pack-reused 69 (from 1)
Receiving objects: 100% (161/161), 1.04 MiB | 14.80 MiB/s, done.
Resolving deltas: 100% (74/74), done.
Note: switching to '213d4d5'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 213d4d5 revert removing webrtcvad
--2026-03-07 13:15:40--  https://github.com/rhass

In [13]:
# 2. Install microWakeWord and dependencies

import locale
import os
import site

def getpreferredencoding(do_setlocale=True):
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

!apt-get -qq update
!apt-get -qq install -y ffmpeg git-lfs
!pip install -q "git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"
!pip install -q "datasets<4" scipy tqdm pyyaml soundfile librosa torchcodec

if not os.path.exists("./micro-wake-word"):
    !git clone https://github.com/OHF-Voice/micro-wake-word

!pip install -q -e ./micro-wake-word
site.main()

import microwakeword
print("microwakeword import OK:", microwakeword.__file__)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.1 MB/s eta 0:00:00 0:00:01
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for microwakeword (pyproject.toml) ... done
microwakeword import OK: /content/micro-wake-word/microwakeword/__init__.py


In [14]:
# 3. Imports

import json
import os
import shutil
import subprocess
from pathlib import Path
from types import SimpleNamespace

import datasets
import numpy as np
import scipy
import yaml
from tqdm import tqdm

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

from generate_samples import generate_samples
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import microwakeword.data as input_data
import microwakeword.mixednet as mixednet
import microwakeword.utils as utils
from microwakeword.layers import modes
from microwakeword.model_train_eval import load_config


In [18]:
# 4. Download RIR dataset

output_dir = Path("./mit_rirs")
source_dir = Path("./MIT_environmental_impulse_responses/16khz")

output_dir.mkdir(exist_ok=True)

if not source_dir.exists():
    !git lfs install
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses

rir_files = list(source_dir.glob("*.wav"))
if not rir_files:
    raise RuntimeError(f"No RIR wav files found in {source_dir}")

existing_rirs = len(list(output_dir.glob("*.wav")))
if existing_rirs != len(rir_files):
    for src in tqdm(rir_files):
        shutil.copy2(src, output_dir / src.name)

print({"mit_rirs": len(list(output_dir.glob('*.wav')))})


{'mit_rirs': 270}


In [5]:
# 5. Audio conversion helper

def convert_audio_tree(src_root, pattern, out_dir):
    src_root = Path(src_root)
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    files = sorted(src_root.glob(pattern))
    if not files:
        raise RuntimeError(f"No source files found in {src_root} matching {pattern}")

    for src in tqdm(files):
        rel_stem = src.relative_to(src_root).with_suffix("")
        dst_name = "__".join(rel_stem.parts) + ".wav"
        dst = out_dir / dst_name
        if dst.exists():
            continue

        subprocess.run(
            ["ffmpeg", "-y", "-i", str(src), "-ac", "1", "-ar", "16000", str(dst)],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

    print({out_dir.name: len(list(out_dir.glob('*.wav')))})


In [6]:
# 6. Download AudioSet sample from parquet and convert to 16 kHz wav

output_dir = Path("./audioset_16k")
output_dir.mkdir(exist_ok=True)
tmp_dir = Path("./audioset_tmp")
tmp_dir.mkdir(exist_ok=True)

max_examples = 2000  # @param {type:"slider", min:500, max:10000, step:500}

audioset = datasets.load_dataset(
    "agkphysics/AudioSet",
    "balanced",
    split=f"train[:{max_examples}]",
)
audioset = audioset.cast_column("audio", datasets.Audio(decode=False))

for row in tqdm(audioset):
    audio = row["audio"]
    dst = output_dir / f"{row['video_id']}.wav"
    if dst.exists():
        continue

    src_path = None
    if audio.get("path") and os.path.exists(audio["path"]):
        src_path = audio["path"]
    elif audio.get("bytes") is not None:
        tmp_src = tmp_dir / f"{row['video_id']}.flac"
        tmp_src.write_bytes(audio["bytes"])
        src_path = str(tmp_src)

    if src_path is None:
        continue

    subprocess.run(
        ["ffmpeg", "-y", "-i", src_path, "-ac", "1", "-ar", "16000", str(dst)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

print({"audioset_16k": len(list(output_dir.glob('*.wav')))})


    
    Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
    You are not authenticated with the Hugging Face Hub in this notebook.
    If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

...

data/eval/34.parquet:   0%|          | 0.00/196M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18683 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17141 [00:00<?, ? examples/s]

100%|██████████| 2000/2000 [05:13<00:00,  6.38it/s]

{'audioset_16k': 2000}


In [7]:
# 7. Download FMA sample and convert to 16 kHz wav

download_dir = Path("./fma")
download_dir.mkdir(exist_ok=True)
fname = "fma_xs.zip"
link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
out_path = download_dir / fname

if not (download_dir / "fma_small").exists():
    if not out_path.exists():
        !wget -O {out_path} {link}
    !cd {download_dir} && unzip -q {fname}

convert_audio_tree("fma/fma_small", "**/*.mp3", "fma_16k")


--2026-03-07 13:32:37--  https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
Resolving huggingface.co (huggingface.co)... 3.166.152.65, 3.166.152.105, 3.166.152.44, ...
Connecting to huggingface.co (huggingface.co)|3.166.152.65|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66ca2060627175be3322dcd0/b15b33649259980016c036a0b2a0fee516e0a9acf3f1adcc561e0abd5efde02b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27fma_xs.zip%3B+filename%3D%22fma_xs.zip%22%3B&response-content-type=application%2Fzip&Expires=1772893957&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzcyODkzOTU3fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZjYTIwNjA2MjcxNzViZTMzMjJkY2QwL2IxNWIzMzY0OTI1OTk4MDAxNmMwMzZhMGIyYTBmZWU1MTZlMGE5YWNmM2YxYWRjYzU2MWUwYWJkNWVmZGUwMmJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eXBlPSoifV19&Signatu

100%|██████████| 210/210 [00:49<00:00,  4.23it/s]

{'fma_16k': 210}


In [8]:
# 8. Generate positive training samples with Piper

os.makedirs("generated_samples", exist_ok=True)

number_of_examples = 3000  # @param {type:"slider", min:500, max:20000, step:500}

generate_samples(
    text=target_word,
    max_samples=number_of_examples,
    output_dir="generated_samples",
    batch_size=32,
    auto_reduce_batch_size=True,
)

print({"generated_samples": len(list(Path('generated_samples').glob('*.wav')))})


{'generated_samples': 3000}


In [23]:
# 9. Build reusable dataset settings and smoke-test the current API

clips_settings = {
    "input_directory": "generated_samples",
    "file_pattern": "*.wav",
    "max_clip_duration_s": None,
    "remove_silence": False,
    "random_split_seed": 10,
    "split_count": 0.1,
}

background_counts = {
    "fma_16k": len(list(Path("fma_16k").rglob("*.wav"))),
    "audioset_16k": len(list(Path("audioset_16k").rglob("*.wav"))),
}
if not all(background_counts.values()):
    raise RuntimeError(f"Background audio is missing. Counts: {background_counts}")

augmentation_settings = {
    "augmentation_duration_s": 3.2,
    "augmentation_probabilities": {
        "SevenBandParametricEQ": 0.1,
        "TanhDistortion": 0.1,
        "PitchShift": 0.1,
        "BandStopFilter": 0.1,
        "AddColorNoise": 0.1,
        "AddBackgroundNoise": 0.75,
        "Gain": 1.0,
        "RIR": 0.5,
    },
    "impulse_paths": ["mit_rirs"],
    "background_paths": ["fma_16k", "audioset_16k"],
    "background_min_snr_db": -5,
    "background_max_snr_db": 10,
    "min_jitter_s": 0.195,
    "max_jitter_s": 0.205,
}

positive_spectrogram_generation_settings = {
    "step_ms": 10,
}

negative_spectrogram_generation_settings = {
    "step_ms": 10,
}

negative_augmentation_settings = {
    "augmentation_duration_s": 3.2,
    "augmentation_probabilities": {
        "SevenBandParametricEQ": 0.0,
        "TanhDistortion": 0.0,
        "PitchShift": 0.0,
        "BandStopFilter": 0.0,
        "AddColorNoise": 0.0,
        "AddBackgroundNoise": 0.0,
        "Gain": 0.0,
        "GainTransition": 0.0,
        "RIR": 0.0,
    },
    "impulse_paths": [],
    "background_paths": [],
    "min_jitter_s": 0.0,
    "max_jitter_s": 0.0,
    "truncate_randomly": True,
}

fma_clips_settings = {
    "input_directory": "fma_16k",
    "file_pattern": "*.wav",
    "max_clip_duration_s": None,
    "remove_silence": False,
    "random_split_seed": 10,
    "split_count": 0.1,
}

audioset_clips_settings = {
    "input_directory": "audioset_16k",
    "file_pattern": "*.wav",
    "max_clip_duration_s": None,
    "remove_silence": False,
    "random_split_seed": 10,
    "split_count": 0.1,
}

class NotebookClips:
    def __init__(
        self,
        input_directory,
        file_pattern,
        min_clip_duration_s=None,
        max_clip_duration_s=None,
        repeat_clip_min_duration_s=None,
        remove_silence=False,
        random_split_seed=None,
        split_count=0.1,
        trimmed_clip_duration_s=None,
        trim_zeros=False,
    ):
        self.repeat_clip_min_duration_s = repeat_clip_min_duration_s or 0.0
        self.trimmed_clip_duration_s = trimmed_clip_duration_s
        self.trim_zeros = trim_zeros
        self.remove_silence = remove_silence
        self.clip_paths = sorted(Path(input_directory).glob(file_pattern))
        if not self.clip_paths:
            raise RuntimeError(f"No audio clips found in {input_directory} matching {file_pattern}")

        min_duration = 0.0 if min_clip_duration_s is None else min_clip_duration_s
        max_duration = float("inf") if max_clip_duration_s is None else max_clip_duration_s
        if min_duration > 0.0 or max_duration != float("inf"):
            filtered = []
            for path in self.clip_paths:
                info = scipy.io.wavfile.read(path, mmap=True)
                sample_rate = info[0]
                samples = info[1].shape[0]
                duration = samples / sample_rate
                if min_duration < duration < max_duration:
                    filtered.append(path)
            self.clip_paths = filtered
        if not self.clip_paths:
            raise RuntimeError("All candidate clips were filtered out before the smoke test.")
        self.clips = self.clip_paths

    def _load_clip(self, path):
        sample_rate, audio = scipy.io.wavfile.read(path)
        audio = np.asarray(audio)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if np.issubdtype(audio.dtype, np.integer):
            audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
        else:
            audio = audio.astype(np.float32)
        if sample_rate != 16000:
            gcd = np.gcd(sample_rate, 16000)
            audio = scipy.signal.resample_poly(audio, 16000 // gcd, sample_rate // gcd).astype(np.float32)
        if self.trim_zeros:
            audio = np.trim_zeros(audio)
        if self.trimmed_clip_duration_s:
            audio = audio[: int(self.trimmed_clip_duration_s * 16000)]
        return self.repeat_clip(audio)

    def repeat_clip(self, audio_samples):
        desired_samples = int(self.repeat_clip_min_duration_s * 16000)
        if desired_samples <= 0 or audio_samples.shape[0] == 0:
            return audio_samples
        original_clip = audio_samples
        while audio_samples.shape[0] < desired_samples:
            audio_samples = np.append(audio_samples, original_clip)
        return audio_samples

    def get_random_clip(self):
        return self._load_clip(self.clip_paths[np.random.randint(len(self.clip_paths))])

    def audio_generator(self, split=None, repeat=1):
        for _ in range(repeat):
            for path in self.clip_paths:
                yield self._load_clip(path)

    def random_audio_generator(self, max_clips=float("inf")):
        while max_clips > 0:
            max_clips -= 1
            yield self.get_random_clip()

clips = NotebookClips(**clips_settings)
augmenter = Augmentation(**augmentation_settings)
spectrograms = SpectrogramGeneration(
    clips=clips,
    augmenter=augmenter,
    **positive_spectrogram_generation_settings,
)

sample_spectrogram = next(spectrograms.spectrogram_generator(random=True, max_clips=1))
print({
    "background_counts": background_counts,
    "sample_spectrogram_shape": sample_spectrogram.shape,
})


{'background_counts': {'fma_16k': 210, 'audioset_16k': 2000}, 'sample_spectrogram_shape': (317, 40)}


In [20]:
# 10. Write full training config

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "trained_models/hey_igor_full"
config["features"] = [
    {
        "type": "clips",
        "truth": True,
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truncation_strategy": "truncate_start",
        "clips_settings": clips_settings,
        "augmentation_settings": augmentation_settings,
        "spectrogram_generation_settings": positive_spectrogram_generation_settings,
    },
    {
        "type": "clips",
        "truth": False,
        "sampling_weight": 8.0,
        "penalty_weight": 1.0,
        "truncation_strategy": "random",
        "clips_settings": fma_clips_settings,
        "augmentation_settings": negative_augmentation_settings,
        "spectrogram_generation_settings": negative_spectrogram_generation_settings,
    },
    {
        "type": "clips",
        "truth": False,
        "sampling_weight": 8.0,
        "penalty_weight": 1.0,
        "truncation_strategy": "random",
        "clips_settings": audioset_clips_settings,
        "augmentation_settings": negative_augmentation_settings,
        "spectrogram_generation_settings": negative_spectrogram_generation_settings,
    },
]
config["training_steps"] = [100]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]
config["learning_rates"] = [0.001]
config["batch_size"] = 16
config["time_mask_max_size"] = [0]
config["time_mask_count"] = [0]
config["freq_mask_max_size"] = [0]
config["freq_mask_count"] = [0]
config["eval_step_interval"] = 999999
config["clip_duration_ms"] = 1500
config["target_minimization"] = 0.9
config["minimization_metric"] = None
config["maximization_metric"] = "average_viable_recall"

with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f)

print(yaml.safe_load(open("training_parameters.yaml"))["train_dir"])


trained_models/hey_igor_full


In [24]:
# 11. Train full model

input_data.Clips = NotebookClips

flags = SimpleNamespace(
    training_config="training_parameters.yaml",
    train=1,
    restore_checkpoint=0,
    test_tf_nonstreaming=0,
    test_tflite_nonstreaming=0,
    test_tflite_nonstreaming_quantized=0,
    test_tflite_streaming=0,
    test_tflite_streaming_quantized=0,
    use_weights="last_weights",
    verbosity="INFO",
    model_name="mixednet",
    pointwise_filters="64,64,64,64",
    repeat_in_block="1,1,1,1",
    mixconv_kernel_sizes="[5],[7,11],[9,15],[23]",
    residual_connection="0,0,0,0",
    first_conv_filters=32,
    first_conv_kernel_size=5,
    stride=3,
    max_pool=0,
    spatial_attention=0,
    pooled=0,
)

config = load_config(flags, mixednet)
shutil.rmtree(config["train_dir"], ignore_errors=True)

data_processor = input_data.FeatureHandler(config)
model = mixednet.model(flags, config["training_input_shape"], config["batch_size"])
print(model.summary())

weights_path = Path(config["train_dir"]) / "last_weights.weights.h5"

try:
    import microwakeword.model_train_eval as model_train_eval
    model_train_eval.train_model(config, model, data_processor, restore_checkpoint=False)
    print("Training completed cleanly.")
except Exception as exc:
    weights_exist = weights_path.exists()
    message = str(exc)
    known_validation_bug = (
        "Invalid input shape for input Tensor" in message
        and "Expected shape" in message
    )
    if weights_exist and known_validation_bug:
        print(
            "Training produced weights, then hit the known clips-based validation bug. "
            "Proceeding with export from last_weights.weights.h5."
        )
    else:
        raise RuntimeError(
            "Training failed before producing usable weights. "
            f"weights_exist={weights_exist}. Original error: {message}"
        ) from exc

print("Training weights:", weights_path)

# If the warm-up run worked, expand to the full run on the same stable batch size.
full_config = yaml.safe_load(open("training_parameters.yaml"))
full_config["training_steps"] = [2000]
with open("training_parameters.yaml", "w") as f:
    yaml.dump(full_config, f)

print("Updated training_parameters.yaml for full run (training_steps=2000, batch_size=16).")
print("Re-run this cell to start the full training run now that the warm-up configuration is stable.")



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (16, 204, 40)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (16, 204, 1, 40)  │          0 │ input_layer[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stream (Stream)     │ (16, 67, 1, 32)   │      6,400 │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (16, 67, 1, 32)   │          0 │ stream[0][0]      │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stream_1 (Stream)   │ (16, 67, 1, 32)   │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d    │ (16, 63, 1, 32)   │        192 │ stream_1[0][0]    │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (16, 63, 1, 64)   │      2,048 │ depthwise_conv2d… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (16, 63, 1, 64)   │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (16, 63, 1, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stream_2 (Stream)   │ (16, 63, 1, 64)   │          0 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ channel_split       │ [(16, 63, 1, 32), │          0 │ stream_2[0][0]    │
│ (ChannelSplit)      │ (16, 63, 1, 32)]  │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ strided_keep        │ (16, 63, 1, 32)   │          0 │ channel_split[0]… │
│ (StridedKeep)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ strided_keep_1      │ (16, 63, 1, 32)   │          0 │ channel_split[0]… │
│ (StridedKeep)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_1  │ (16, 57, 1, 32)   │        256 │ strided_keep[0][… │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_2  │ (16, 53, 1, 32)   │        384 │ strided_keep_1[0… │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ strided_drop        │ (16, 53, 1, 32)   │          0 │ depthwise_conv2d… │
│ (StridedDrop)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ strided_drop_1      │ (16, 53, 1, 32)   │          0 │ depthwise_conv2d… │
│ (StridedDrop)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (16, 53, 1, 64)   │          0 │ strided_drop[0][… │
│ (Concatenate)       │                   │            │ strided_drop_1[0

 Total params: 26,049 (101.75 KB)

 Trainable params: 25,537 (99.75 KB)

 Non-trainable params: 512 (2.00 KB)

None


Training produced weights, then hit the known clips-based validation bug. Proceeding with export from last_weights.weights.h5.
Training weights: trained_models/hey_igor_full/last_weights.weights.h5
Updated training_parameters.yaml for full run (training_steps=2000, batch_size=16).
Re-run this cell to start the full training run now that the warm-up configuration is stable.


In [25]:
# 12. Export quantized streaming TFLite

flags = SimpleNamespace(
    training_config="training_parameters.yaml",
    stride=3,
    pointwise_filters="64,64,64,64",
    repeat_in_block="1,1,1,1",
    mixconv_kernel_sizes="[5],[7,11],[9,15],[23]",
    residual_connection="0,0,0,0",
    first_conv_filters=32,
    first_conv_kernel_size=5,
    spatial_attention=0,
    max_pool=0,
    pooled=0,
)

config = load_config(flags, mixednet)
audio_processor = input_data.FeatureHandler(config)

model = mixednet.model(flags, shape=config["training_input_shape"], batch_size=1)
model.load_weights(os.path.join(config["train_dir"], "last_weights.weights.h5"))

utils.convert_model_saved(
    model,
    config,
    folder="stream_state_internal",
    mode=modes.Modes.STREAM_INTERNAL_STATE_INFERENCE,
)

utils.convert_saved_model_to_tflite(
    config,
    audio_processor=audio_processor,
    path_to_model=os.path.join(config["train_dir"], "stream_state_internal"),
    folder=os.path.join(config["train_dir"], "tflite_stream_state_internal_quant"),
    fname="stream_state_internal_quant.tflite",
    quantize=True,
)

print("TFLite artifacts:")
for p in sorted(Path(config["train_dir"]).rglob("*.tflite")):
    print("-", p)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_audio         │ (1, 3, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims_2       │ (1, 3, 1, 40)     │          0 │ input_audio[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stream_6 (Stream)   │ (1, 1, 1, 32)     │      6,480 │ expand_dims_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (1, 1, 1, 32)     │          0 │ stream_6[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stream_7 (Stream)   │ (1, 5, 1, 32)     │        128 │ activation_5[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_6  │ (1, 1, 1, 32)     │        192 │ stream_7[0][0]    │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (1, 1, 1, 64)     │      2,048 │ depthwise_conv2d… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (1, 1, 1, 64)     │        256 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (1, 1, 1, 64)     │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stream_8 (Stream)   │ (1, 11, 1, 64)    │        640 │ activation_6[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ channel_split_2     │ [(1, 11, 1, 32),  │          0 │ stream_8[0][0]    │
│ (ChannelSplit)      │ (1, 11, 1, 32)]   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ strided_keep_4      │ (1, 7, 1, 32)     │          0 │ channel_split_2[… │
│ (StridedKeep)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ strided_keep_5      │ (1, 11, 1, 32)    │          0 │ channel_split_2[… │
│ (StridedKeep)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_7  │ (1, 1, 1, 32)     │        256 │ strided_keep_4[0… │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_8  │ (1, 1, 1, 32)     │        384 │ strided_keep_5[0… │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ strided_drop_4      │ (1, 1, 1, 32)     │          0 │ depthwise_conv2d… │
│ (StridedDrop)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ strided_drop_5      │ (1, 1, 1, 32)     │          0 │ depthwise_conv2d… │
│ (StridedDrop)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (1, 1, 1, 64)     │          0 │ strided_drop_4[0… │
│ (Concatenate)       │                   │            │ strided_drop_5[0

 Total params: 30,225 (118.07 KB)

 Trainable params: 25,537 (99.75 KB)

 Non-trainable params: 4,688 (18.31 KB)

Saved artifact at 'trained_models/hey_igor_full/stream_state_internal'. The following endpoints are available:

* Endpoint 'serve'
  inputs (POSITIONAL_OR_KEYWORD): TensorSpec(shape=(1, 3, 40), dtype=tf.float32, name=None)
  training (POSITIONAL_OR_KEYWORD): Literal[None]
  mask (POSITIONAL_OR_KEYWORD): Literal[None]
Output Type:
  TensorSpec(shape=(1, 1), dtype=tf.float32, name=None)
Captures:
  135070552716688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135069928549328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135070552716304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135070552717264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135070552718800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135070552715152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135070552715344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135070552716112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135070552719184: TensorSpec(shape=(), dt

TFLite artifacts:
- trained_models/hey_igor_full/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite


In [27]:
# 13. Package ESPHome model files

from google.colab import files

src_model = Path("trained_models/hey_igor_full/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")
if not src_model.exists():
    raise RuntimeError(f"Missing model: {src_model}")

shutil.copy(src_model, "hey_igor.tflite")

manifest = {
    "type": "micro",
    "wake_word": "Hey Igor",
    "author": "Alex",
    "website": "https://github.com/cajo-dk/microwakeword",
    "model": "hey_igor.tflite",
    "trained_languages": ["en"],
    "version": 2,
    "micro": {
        "probability_cutoff": 0.97,
        "sliding_window_size": 5,
        "feature_step_size": 10,
        "tensor_arena_size": 262144,
        "minimum_esphome_version": "2024.7",
    },
}

with open("hey_igor.json", "w") as f:
    json.dump(manifest, f, indent=2)

files.download("hey_igor.tflite")
files.download("hey_igor.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>